# Credit Decisioning – Data Understanding

## Objective
Understand and combine loan application datasets (train + test) to prepare a unified dataset
for business analysis related to loan approval and default risk.

At this stage, **no modeling or heavy cleaning** is performed.
The goal is to understand data structure, completeness, and initial risks.


In [1]:
import pandas as pd
import numpy as np


In [10]:
train_df = pd.read_csv("../data/TrainingData.csv")
test_df = pd.read_csv("../data/TestData.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (252000, 13)
Test shape: (28000, 12)


We are provided with two datasets:
- **Train dataset**: Contains historical loan applications with outcomes
- **Test dataset**: Contains new loan applications (may miss outcome variables)

For analytics, both datasets will be combined to reflect real-world incoming data.


In [11]:
print("Train columns:")
print(train_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())


Train columns:
['Id', 'Income', 'Age', 'Experience', 'Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE', 'CURRENT_JOB_YRS', 'CURRENT_HOUSE_YRS', 'Risk_Flag']

Test columns:
['ID', 'Income', 'Age', 'Experience', 'Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE', 'CURRENT_JOB_YRS', 'CURRENT_HOUSE_YRS']


In [12]:
train_df.columns = train_df.columns.str.lower().str.strip()
test_df.columns = test_df.columns.str.lower().str.strip()

print("Standardized Train columns:")
print(train_df.columns.tolist())

print("\nStandardized Test columns:")
print(test_df.columns.tolist())


Standardized Train columns:
['id', 'income', 'age', 'experience', 'married/single', 'house_ownership', 'car_ownership', 'profession', 'city', 'state', 'current_job_yrs', 'current_house_yrs', 'risk_flag']

Standardized Test columns:
['id', 'income', 'age', 'experience', 'married/single', 'house_ownership', 'car_ownership', 'profession', 'city', 'state', 'current_job_yrs', 'current_house_yrs']


Column names are standardized to lowercase for consistency.
Any differences between train and test columns will be handled explicitly.


In [13]:
train_only_cols = set(train_df.columns) - set(test_df.columns)
test_only_cols = set(test_df.columns) - set(train_df.columns)

print("Columns only in train:", train_only_cols)
print("Columns only in test:", test_only_cols)


Columns only in train: {'risk_flag'}
Columns only in test: set()


### Target Columns Interpretation
- `loan_status`: Indicates whether a loan was approved or rejected
- `default`: Indicates whether the borrower defaulted on repayment

It is expected that:
- `default` may exist **only in train data**
- Missing values in test data are realistic and will be preserved


In [14]:
train_df["source"] = "train"
test_df["source"] = "test"


In [15]:
combined_columns = sorted(set(train_df.columns).union(set(test_df.columns)))

train_df = train_df.reindex(columns=combined_columns)
test_df = test_df.reindex(columns=combined_columns)


loan_df = pd.concat([train_df, test_df], ignore_index=True)

print("Combined dataset shape:", loan_df.shape)




Combined dataset shape: (280000, 14)


The datasets have been combined into a single table with a `source` column.
This reflects a real business scenario where historical and new applications coexist.


In [16]:
loan_df.info()
loan_df.describe(include="all")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280000 entries, 0 to 279999
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   age                280000 non-null  int64  
 1   car_ownership      280000 non-null  object 
 2   city               280000 non-null  object 
 3   current_house_yrs  280000 non-null  int64  
 4   current_job_yrs    280000 non-null  int64  
 5   experience         280000 non-null  int64  
 6   house_ownership    280000 non-null  object 
 7   id                 280000 non-null  int64  
 8   income             280000 non-null  int64  
 9   married/single     280000 non-null  object 
 10  profession         280000 non-null  object 
 11  risk_flag          252000 non-null  float64
 12  source             280000 non-null  object 
 13  state              280000 non-null  object 
dtypes: float64(1), int64(6), object(7)
memory usage: 29.9+ MB


,age,car_ownership,city,current_house_yrs,current_job_yrs,experience,house_ownership,id,income,married/single,profession,risk_flag,source,state
count,280000.000000,280000,280000,280000.000000,280000.000000,280000.000000,280000,280000.000000,2.800000e+05,280000,280000,252000.000000,280000,280000
unique,NaN,2,333,NaN,NaN,NaN,3,NaN,NaN,2,74,NaN,2,37
top,NaN,no,Vijayanagaram,NaN,NaN,NaN,rented,NaN,NaN,single,Physician,NaN,train,Uttar_Pradesh
freq,NaN,195625,1407,NaN,NaN,NaN,257703,NaN,NaN,251442,6587,NaN,252000,28400
mean,49.964132,NaN,NaN,11.997193,6.334418,10.088032,NaN,114800.500000,5.000361e+06,NaN,NaN,0.123000,NaN,NaN
std,17.070465,NaN,NaN,1.398907,3.646864,6.005066,NaN,76800.484365,2.876988e+06,NaN,NaN,0.328438,NaN,NaN
min,21.000000,NaN,NaN,10.000000,0.000000,0.000000,NaN,1.000000,1.031000e+04,NaN,NaN,0.000000,NaN,NaN
25%,35.000000,NaN,NaN,11.000000,3.000000,5.000000,NaN,42000.750000,2.506726e+06,NaN,NaN,0.000000,NaN,NaN
50%,50.000000,NaN,NaN,12.000000,6.000000,10.000000,NaN,112000.500000,5.003310e+06,NaN,NaN,0.000000,NaN,NaN
75%,65.000000,NaN,NaN,13.000000,9.000000,15.000000,NaN,182000.250000,7.477502e+06,NaN,NaN,0.000000,NaN,NaN


In [17]:
missing_values = loan_df.isnull().sum().sort_values(ascending=False)
missing_percentage = (missing_values / len(loan_df)) * 100

missing_df = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percentage": missing_percentage
})

missing_df[missing_df["missing_count"] > 0]

if "loan_status" in loan_df.columns:
    loan_df["loan_status"].value_counts(dropna=False)

if "default" in loan_df.columns:
    loan_df["default"].value_counts(dropna=False)


In [18]:
loan_df.to_csv("../data/loan_data_combined.csv", index=False)
print("Combined dataset saved to data/loan_data_combined.csv")


Combined dataset saved to data/loan_data_combined.csv
